# Leçon 11 - Protocole Agent-à-Agent (A2A)


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Qu'est-ce que le protocole A2A ?

Le **protocole Agent-to-Agent (A2A)** est une norme ouverte qui permet aux agents IA de communiquer,
de se découvrir mutuellement et de collaborer — même s’ils sont construits sur des frameworks différents ou hébergés
par des services différents.

Concepts clés :

- **Découverte** – Les agents publient une *Fiche Agent* qui décrit leurs capacités, facilitant ainsi
  la tâche pour que d’autres agents (ou des orchestres) trouvent le spécialiste adapté à une tâche.
- **Échange de messages** – Les agents échangent des messages structurés via un protocole commun, de sorte qu’une
  requête d’un agent puisse être comprise et exécutée par un autre, quel que soit son implementation
  interne.
- **Cycle de vie des tâches** – A2A définit des états tels que *soumis*, *en cours*, *terminé* et
  *échoué*, donnant à l’orchestrateur une visibilité complète sur la progression d’une tâche déléguée.

Dans cette leçon, nous simulons une collaboration de type A2A en connectant trois agents spécialisés en voyages
dans un flux de travail où chaque agent apporte son expertise et passe les résultats au suivant.


## Création d’agents de voyage spécialisés


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Multi-Agent Collaboration via Workflow

We connect the three agents into a sequential workflow that mirrors A2A message passing:

1. **CurrencyExchangeAgent** receives the user request and produces currency guidance.
2. **ActivityPlannerAgent** receives the enriched context and adds activity recommendations.
3. **TravelManagerAgent** synthesizes both inputs into a final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendre A2A en production

Dans un environnement de production, le protocole A2A permet des scénarios inter-services puissants :

| Fonctionnalité | Description |
|---|---|
| **Interopérabilité inter-framework** | Un agent construit avec un framework peut déléguer des tâches à un agent construit avec n'importe quel autre framework compatible A2A, permettant une véritable interopérabilité entre organisations. |
| **Limites de service** | Les agents peuvent vivre dans des microservices séparés, des régions cloud ou même dans des organisations différentes tout en collaborant sans interruption. |
| **Découverte dynamique** | Un orchestrateur peut interroger un registre de Cartes d'Agent au moment de l'exécution pour trouver le spécialiste le mieux adapté à une sous-tâche donnée. |
| **Streaming & notifications push** | A2A supporte les Server-Sent Events (SSE) pour les mises à jour en temps réel de la progression et les notifications push pour les tâches longues. |

Le workflow que nous avons construit ci-dessus est une version simplifiée et en processus de ce modèle. Dans une vraie
déploiement, chaque agent exposerait un point de terminaison HTTP, publierait une Carte d'Agent, et communiquerait
via le protocole A2A JSON-RPC.


## Résumé

Dans cette leçon, vous avez appris :

1. **Ce qu'est le protocole A2A** — une norme ouverte pour la découverte, la messagerie,
   et la gestion des tâches entre agents.
2. **Comment créer des agents spécialisés** — un agent d'échange de devises, un agent planificateur d'activités,
   et un orchestrateur gestionnaire de voyages.
3. **Comment connecter les agents dans un flux de travail** — en utilisant `WorkflowBuilder` pour modéliser la transmission séquentielle
   de messages entre agents.
4. **Comment A2A fonctionne en production** — permettant une collaboration inter-framework et inter-service
   avec une découverte dynamique et des mises à jour en continu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
